In [1]:
import pybedtools
import numpy as np
import pandas as pd
import tensorflow as tf
import pyfaidx
import os
import sys
import pyBigWig
sys.path.append("..")
from fibr_ipsc_peaks import helpers
from utils import sequence
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import importlib
# set TF_GPU_ALLOCATOR=cuda_malloc_async
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
import tensorflow as tf

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)



The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


/tmp/ipykernel_2959173/243792528.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
2024-04-03 03:58:05.218422: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-03 03:58:10.365077: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-03 03:5

1 Physical GPUs, 1 Logical GPUs


2024-04-03 03:59:39.124206: I tensorflow/core/common_runtime/gpu/gpu_process_state.cc:236] Using CUDA malloc Async allocator for GPU: 0
2024-04-03 03:59:39.126061: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 54913 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe MIG 7g.80gb, pci bus id: 0000:ca:00.0, compute capability: 8.0


In [2]:
hg38 = pyfaidx.Fasta('/users/surag/genomes/hg38/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta')

data_dir = "/oak/stanford/groups/akundaje/vir/tfatlas/processed_data/"

In [3]:
oct4_esc = "ENCSR000BMU"
ap1_fibroblast = "ENCSR124AIG"

peaks_file = "peaks_inliers.bed.gz"
background_peaks = "gc_neg_only.bed.gz"

esc_peaks_file = os.path.join(data_dir, oct4_esc, peaks_file)
fibroblast_peaks_file = os.path.join(data_dir, ap1_fibroblast, peaks_file)

esc_peaks_file = os.path.join("/users/kobbad/feat_leakage/data/bed_files/fibro_ipsc", "ENCSR000BMU_peaks_inliers.bed.gz")
fibroblast_peaks_file = os.path.join("/users/kobbad/feat_leakage/data/bed_files/fibro_ipsc", "ENCSR124AIG_peaks_inliers.bed.gz")

esc_peaks = pybedtools.BedTool(os.path.join(data_dir, oct4_esc, peaks_file))
fibroblast_peaks = pybedtools.BedTool(os.path.join(data_dir, ap1_fibroblast, peaks_file))

esc_background = pybedtools.BedTool(os.path.join(data_dir, oct4_esc, background_peaks)).to_dataframe()
fibroblast_background = pybedtools.BedTool(os.path.join(data_dir, ap1_fibroblast, background_peaks)).to_dataframe()



In [4]:
# bed file has merged peaks from both ipsc and fibroblast
fibro_ipsc_peaks = pd.read_csv("labeled_peaks.bed", sep="\t", header=0)

# read in labeled peaks 
labels = np.load("labeled_peaks.npy")

In [5]:
# create train, val, test split

X_train, X_val, X_test, y_train, y_val, y_test = helpers.create_train_val_test(hg38, fibro_ipsc_peaks, labels)

In [14]:
multiask_model = tf.keras.models.load_model("models/unbalanced_single_task_model.h5")

In [16]:
x = multiask_model.predict(X_train)

819/819 [==============================] - 5s 6ms/step


In [17]:
x.shape

(26195, 1)

In [13]:
seqs = np.concatenate([X_train, X_val, X_test])
seqs.shape

(40931, 300, 4)

In [19]:
# make seqs a tensor
seqs = tf.convert_to_tensor(seqs, dtype=tf.float32)

with tf.GradientTape() as tape:
    # Watch the input tensor
    tape.watch(seqs)
    
    # Forward pass through the model with batched inputs
    outputs = multiask_model(seqs)
    


2024-03-30 01:08:20.608520: W tensorflow/core/kernels/gpu_utils.cc:54] Failed to allocate memory for convolution redzone checking; skipping this check. This is benign and only means that we won't check cudnn for out-of-bounds reads and writes. This message will only be printed once.


In [20]:
grads = tape.gradient(outputs, seqs)

In [23]:
grad_x_input = grads * seqs

In [24]:
grad_x_input.shape

TensorShape([40931, 300, 4])

In [29]:
# save gradxinput to unbalanced_single_task_model_gradxinput.npz
np.savez("unbalanced_single_task_model_gradxinput.npz", grad_x_input.numpy())


In [30]:
# save seqs to unbalanced_single_task_model_seqs.npz
np.savez("unbalanced_single_task_model_seqs.npz", seqs.numpy())

In [35]:
seqs = seqs.numpy()
grad_x_input = grad_x_input.numpy()

In [37]:
seqs = seqs.transpose(0, 2, 1)
grad_x_input = grad_x_input.transpose(0, 2, 1)

In [39]:
np.savez("unbalanced_single_task_model_gradxinput.npz", grad_x_input)
# save seqs to unbalanced_single_task_model_seqs.npz
np.savez("unbalanced_single_task_model_seqs.npz", seqs)


In [33]:
y['arr_0'].shape

(40931, 300, 4)

### balanced single task

In [6]:
balanced_model = tf.keras.models.load_model("models/balanced_single_task_model.h5")

In [24]:
x = balanced_model.predict(X_train)

819/819 [==============================] - 5s 6ms/step


In [25]:
seqs = np.concatenate([X_train, X_val, X_test])
seqs.shape

(40931, 300, 4)

In [26]:
# make seqs a tensor
seqs = tf.convert_to_tensor(seqs, dtype=tf.float32)

with tf.GradientTape() as tape:
    # Watch the input tensor
    tape.watch(seqs)
    
    # Forward pass through the model with batched inputs
    outputs = balanced_model(seqs)
    


In [27]:
grads = tape.gradient(outputs, seqs)

2024-04-03 04:42:59.344690: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:306] gpu_async_0 cuMemAllocAsync failed to allocate 2981418000 bytes: CUDA error: out of memory (CUDA_ERROR_OUT_OF_MEMORY)
 Reported by CUDA: Free memory/Total memory: 1584660480/84987740160
2024-04-03 04:42:59.344757: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:311] Stats: Limit:                     57580716032
InUse:                     48306091300
MaxInUse:                  58407923804
NumAllocs:                       76440
MaxAllocSize:               7541800056
Reserved:                            0
PeakReserved:                        0
LargestFreeBlock:                    0

2024-04-03 04:42:59.344791: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:63] Histogram of current allocation: (allocation_size_in_bytes, nb_allocation_of_that_sizes), ...;
2024-04-03 04:42:59.344801: E external/local_xla/xla/stream_executo

In [28]:
grad_x_input = grads * seqs

In [29]:
seqs = seqs.numpy()
grad_x_input = grad_x_input.numpy()
seqs = seqs.transpose(0, 2, 1)
grad_x_input = grad_x_input.transpose(0, 2, 1)

In [30]:
np.savez("balanced_single_task_model_gradxinput.npz", grad_x_input)
# save seqs to unbalanced_single_task_model_seqs.npz
np.savez("balanced_single_task_model_seqs.npz", seqs)


In [14]:
grad_x_input.shape

TensorShape([40931, 300, 4])

### multitask model

In [15]:
multiask_model = tf.keras.models.load_model("models/balanced_multitask_model.h5")

In [16]:
x = multiask_model.predict(X_train)

819/819 [==============================] - 6s 7ms/step


In [17]:
seqs = np.concatenate([X_train, X_val, X_test])
seqs.shape

(40931, 300, 4)

In [18]:
# make seqs a tensor
seqs = tf.convert_to_tensor(seqs, dtype=tf.float32)

with tf.GradientTape() as tape:
    # Watch the input tensor
    tape.watch(seqs)
    
    # Forward pass through the model with batched inputs
    outputs = multiask_model(seqs)
    


2024-04-03 04:38:35.724992: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:306] gpu_async_0 cuMemAllocAsync failed to allocate 6807853072 bytes: CUDA error: out of memory (CUDA_ERROR_OUT_OF_MEMORY)
 Reported by CUDA: Free memory/Total memory: 1584660480/84987740160
2024-04-03 04:38:35.725027: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:311] Stats: Limit:                     57580716032
InUse:                     52664053872
MaxInUse:                  58407923804
NumAllocs:                       51291
MaxAllocSize:               7541800056
Reserved:                            0
PeakReserved:                        0
LargestFreeBlock:                    0

2024-04-03 04:38:35.725042: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:63] Histogram of current allocation: (allocation_size_in_bytes, nb_allocation_of_that_sizes), ...;
2024-04-03 04:38:35.725046: E external/local_xla/xla/stream_executo

In [19]:
grads = tape.gradient(outputs, seqs)

2024-04-03 04:39:57.006921: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:306] gpu_async_0 cuMemAllocAsync failed to allocate 4545689008 bytes: CUDA error: out of memory (CUDA_ERROR_OUT_OF_MEMORY)
 Reported by CUDA: Free memory/Total memory: 1584660480/84987740160
2024-04-03 04:39:57.006972: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:311] Stats: Limit:                     57580716032
InUse:                     53044128784
MaxInUse:                  58407923804
NumAllocs:                       51650
MaxAllocSize:               7541800056
Reserved:                            0
PeakReserved:                        0
LargestFreeBlock:                    0

2024-04-03 04:39:57.007000: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:63] Histogram of current allocation: (allocation_size_in_bytes, nb_allocation_of_that_sizes), ...;
2024-04-03 04:39:57.007004: E external/local_xla/xla/stream_executo

In [20]:
grad_x_input = grads * seqs

In [21]:
seqs = seqs.numpy()
grad_x_input = grad_x_input.numpy()

In [22]:
seqs = seqs.transpose(0, 2, 1)
grad_x_input = grad_x_input.transpose(0, 2, 1)

In [23]:
np.savez("unbalanced_multi_task_model_gradxinput.npz", grad_x_input)
# save seqs to unbalanced_single_task_model_seqs.npz
np.savez("unbalanced_multi_task_model_seqs.npz", seqs)
